## Implementazione di un Bot Discord per la Classificazione CIFAR-10
In questa sezione integriamo il modello addestrato con un Bot Discord. 
Il bot riceverà un'immagine, la processerà e restituirà la classe predetta.

In [ ]:
# Installazione delle librerie necessarie
#!pip install discord.py tensorflow pillow nest-asyncio

import nest_asyncio
import io
import numpy as np
import tensorflow as tf
from PIL import Image
import discord
from discord.ext import commands
import os
from dotenv import load_dotenv 

# Patch necessaria per far girare il loop di Discord dentro Jupyter
nest_asyncio.apply()  #Ho incluso nest_asyncio perché i file Jupyter hanno spesso conflitti con i loop di Discord.

In [ ]:
import os
import discord
from discord.ext import commands
from dotenv import load_dotenv 

load_dotenv('TOKEN')
TOKEN = os.getenv('TOKEN')

##### 1. Caricamento del Modello e Definizione Classi
Carichiamo il file `.h5` generato durante la fase di training e definiamo le 10 etichette del dataset CIFAR-10.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

MODEL_PATH = r'cifar10_improved_model.keras'


try:
    # Aggiungiamo compile=False per ignorare l'ottimizzatore PyTorch incompatibile
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    print("✅ Modello caricato correttamente (modalità inferenza)!")
    
    # Visualizziamo la struttura per essere sicuri che sia integro
    model.summary()
    
except Exception as e:
    print(f"❌ Errore nel caricamento: {e}")

class_names = ['aereo', 'automobile', 'uccello', 'gatto', 'cervo', 
               'cane', 'rana', 'cavallo', 'nave', 'camion']

NameError: name '__file__' is not defined

# Preprocessing

## 2. Funzione di Pre-processing
Le immagini inviate su Discord possono avere qualsiasi dimensione. La funzione `prepare_image` si occupa di:
1. Convertire l'immagine in RGB.
2. Ridimensionarla a 32x32 pixel.
3. Normalizzare i valori dei pixel tra 0 e 1.

In [ ]:
'''from PIL import Image
def prepare_image(image_bytes):
    # .convert('RGB') elimina la trasparenza che confonde il modello
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    img = img.resize((32, 32))
    # Trasformiamo in float32 PRIMA della divisione per la massima precisione
    img_array = np.array(img).astype('float32') / 255.0
    return np.expand_dims(img_array, axis=0)'''

In [ ]:
'''def prepare_image(image_bytes):
    # Conversione in PNG prima del processamento (Standardizzazione del segnale)
    img_temp = Image.open(io.BytesIO(image_bytes))
    with io.BytesIO() as png_buffer:
        img_temp.save(png_buffer, format='PNG')
        png_buffer.seek(0)
        # Procediamo con l'immagine standardizzata in RGB
        img = Image.open(png_buffer).convert('RGB')
    
    img = img.resize((32, 32))
    # Normalizzazione: portiamo i pixel nel range [0, 1]
    img_array = np.array(img).astype('float32') / 255.0
    # Aggiungiamo la dimensione del batch (1, 32, 32, 3)
    return np.expand_dims(img_array, axis=0)'''

In [ ]:
def prepare_image(image_bytes):
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    img = img.resize((32, 32), Image.Resampling.LANCZOS)
    # Normalizzazione [0, 1] coerente con la cella 5 di CNN_base
    img_array = np.array(img).astype('float32') / 255.0
    return np.expand_dims(img_array, axis=0)

# Configurazione e Comandi Bot

### 3. Configurazione del Bot Discord
Definiamo il prefisso del comando (`!`) e la logica per gestire l'allegato. 
Il bot scaricherà l'immagine, userà il modello per la predizione e risponderà all'utente.

In [ ]:
# Configurazione permessi (Intents)
intents = discord.Intents.default()
intents.message_content = True 
bot = commands.Bot(command_prefix="!", intents=intents)

In [ ]:
'''import discord
from discord.ext import commands
import tensorflow as tf
import numpy as np
import io
from PIL import Image

# Configurazione permessi (Intents)
intents = discord.Intents.default()
intents.message_content = True 
bot = commands.Bot(command_prefix="!", intents=intents)


# 1. FUNZIONE DI SUPPORTO (Il Pre-processore)


# 2. RESET DEL SISTEMA (Per evitare errori in Jupyter)
#bot.remove_command('classifica')

# 3. IL COMANDO INTEGRATO (Cuore + Grafica)
@bot.command()
async def classifica(ctx):
    # Controllo preliminare: c'è un'immagine?
    if not ctx.message.attachments:
        await ctx.send("⚠️ Errore di input: Per favore, allega una foto al comando `!classifica`.")
        return

    attachment = ctx.message.attachments[0]
    
    # Filtro di frequenza (estensioni ammesse)
    if not any(attachment.filename.lower().endswith(ext) for ext in ['png', 'jpg', 'jpeg']):
        await ctx.send("❌ Formato non riconosciuto. Il sistema accetta solo JPG o PNG.")
        return

    # --- INIZIO FASE DINAMICA (Caricamento + Elaborazione) ---
    loading_msg = await ctx.send("⌛ [░░░░░░░░░░] 0% - Ricezione pacchetti dati...")
    
    try:
        # FASE A: Download e lettura (40%)
        image_bytes = await attachment.read()
        await loading_msg.edit(content="⌛ [████░░░░░░] 40% - Elaborazione segnale (Resize & Normalization)...")
        
        # FASE B: Preprocessing (70%)
        processed_img = prepare_image(image_bytes)
        await loading_msg.edit(content="⌛ [███████░░░] 70% - Interrogazione Rete Neurale CNN...")
        
        # FASE C: Inferenza (100%)
        # verbose=0 serve per non sporcare i log del server'''
'''preds = model.predict(processed_img, verbose=0)
        await loading_msg.edit(content="⌛ [██████████] 100% - Analisi completata!")
        
        # Calcolo statistico dei risultati
        index = np.argmax(preds)
        # Applichiamo Softmax per la distribuzione di probabilità
        probs = tf.nn.softmax(preds[0])
        accuracy = np.max(probs) * 100
        label = class_names[index]'''
        # Forza training=False per disattivare il Data Augmentation
'''preds = model(processed_img, training=False) 

        # Converti subito tutto in probabilità
        probs = tf.nn.softmax(preds[0]).numpy()
        index = np.argmax(probs)
        accuracy = probs[index] * 100
        label = class_names[index]

        # --- OUTPUT FINALE (Embed elegante) ---
        embed = discord.Embed(
            title="🔍 Risultato del Riconoscimento",
            description=f"Analisi completata per {ctx.author.mention}",
            color=0x3498db 
        )
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Oggetto Identificato", value=f"✨ **{label.upper()}**", inline=True)
        embed.add_field(name="Grado di Sicurezza", value=f"📈 {accuracy:.2f}%", inline=True)
        embed.set_footer(text="AI Model: CIFAR-10 CNN | MAGLGGICMADFMN")

        # Pulizia: rimuoviamo la barra di caricamento e inviamo il risultato finale
        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await loading_msg.edit(content=f"❌ Errore critico di sistema: {e}")'''

In [ ]:
'''@bot.command()
async def classifica(ctx):
    if not ctx.message.attachments:
        await ctx.send("⚠️ Carica una foto e scrivi `!classifica`!")
        return

    attachment = ctx.message.attachments[0]
    loading_msg = await ctx.send("⌛ [░░░░░░░░░░] Analisi in corso...")
    
    try:
        image_bytes = await attachment.read()
        processed_img = prepare_image(image_bytes)
        print(f"DEBUG - Somma pixel: {np.sum(processed_img)}") 
        print(f"DEBUG - Primi 5 valori: {processed_img[0, 0, 0, :5]}")
        
        # 1. PREDIZIONE: Forza training=False per ignorare il Data Augmentation
        preds = model(processed_img, training=False)
        
        # 2. CALCOLO FISICO DELLE PROBABILITÀ (Softmax)
        probs = tf.nn.softmax(preds[0]).numpy()
        
        # 3. IDENTIFICAZIONE TOP 3
        # Prendiamo gli indici dei 3 valori più alti
        top_3_indices = np.argsort(probs)[-3:][::-1]
        
        # Costruiamo la stringa per l'Embed
        testo_top3 = ""
        for i in top_3_indices:
            nome_classe = class_names[i].capitalize()
            percentuale = probs[i] * 100
            testo_top3 += f"• **{nome_classe}**: {percentuale:.2f}%\n"

        # 4. CREAZIONE EMBED DETTAGLIATO
        index_primo = top_3_indices[0]
        label = class_names[index_primo]
        confidenza = probs[index_primo] * 100

        embed = discord.Embed(
            title="🔍 Analisi Rete Neurale (CIFAR-10)",
            description=f"Dati elaborati per {ctx.author.mention}",
            color=0x3498db
        )
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Risultato Principale", value=f"✨ **{label.upper()}**", inline=False)
        embed.add_field(name="Grado di Sicurezza", value=f"📈 {confidenza:.2f}%", inline=True)
        
        # QUESTA È LA PARTE CHE MANCAVA:
        embed.add_field(name="📊 Top 3 Probabilità", value=testo_top3, inline=False)
        
        embed.set_footer(text="Model: Improved CNN | Physics-based inference")

        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await loading_msg.edit(content=f"❌ Errore tecnico: {e}")'''

In [ ]:
'''@bot.command()
async def classifica(ctx):
    if not ctx.message.attachments:
        await ctx.send("⚠️ Carica una foto!")
        return

    attachment = ctx.message.attachments[0]
    
    # Messaggio di attesa per l'utente
    loading_msg = await ctx.send("⌛ [░░░░░░░░░░] Analisi in corso...")
    
    try:
        # 1. ACQUISIZIONE E PRE-ELABORAZIONE
        image_bytes = await attachment.read()
        processed_img = prepare_image(image_bytes)
        
        # Calcolo diagnostico del segnale
        pixel_sum = np.sum(processed_img)
        
        # 2. INFERENZA
        # training=False è fondamentale perché il tuo modello ha layer di augmentation
        preds = model(processed_img, training=False)
        
        # 3. ELABORAZIONE RISULTATI
        # Il tuo modello ha già 'softmax' come activation nell'ultimo layer
        # Prendiamo direttamente i valori predetti (che sono già probabilità)
        probs = preds[0].numpy() 
        
        # Identifichiamo i 3 risultati migliori
        top_3_indices = np.argsort(probs)[-3:][::-1]
        
        testo_top3 = ""
        for i in top_3_indices:
            nome = class_names[i].upper()
            percentuale = probs[i] * 100
            testo_top3 += f"• **{nome}**: {percentuale:.2f}%\n"

        # 4. CREAZIONE DELL'EMBED FINALE
        index_best = top_3_indices[0]
        label_best = class_names[index_best]
        confidenza_best = probs[index_best] * 100

        embed = discord.Embed(
            title="🔍 Analisi Tecnica Rete Neurale",
            description=f"Risultati per {ctx.author.mention}",
            color=0x3498db
        )
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Predizione Principale", value=f"✨ **{label_best.upper()}**", inline=False)
        embed.add_field(name="Grado di Sicurezza", value=f"📈 {confidenza_best:.2f}%", inline=True)
        embed.add_field(name="📊 Distribuzione Probabilità", value=testo_top3, inline=False)
        embed.set_footer(text=f"Check sum: {pixel_sum:.2f} | Info: Softmax integrata")

        # Rimuoviamo il caricamento e inviamo il risultato
        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await loading_msg.edit(content=f"❌ Errore durante l'analisi: {e}")'''

In [ ]:
#kkkclass_names = ['aereo', 'automobile', 'uccello', 'gatto', 'cervo', 'cane', 'rana', 'cavallo', 'nave', 'camion']

In [ ]:
@bot.command()
async def classifica(ctx):
    if not ctx.message.attachments:
        await ctx.send("⚠️ Allega una foto!")
        return

    attachment = ctx.message.attachments[0]
    # Messaggio di caricamento per feedback immediato
    loading_msg = await ctx.send("⌛ Analisi in corso...")
    
    try:
        image_bytes = await attachment.read()
        processed_img = prepare_image(image_bytes)
        
        # INFERENZA: training=False disattiva RandomRotation/Flip del modello
        preds = model(processed_img, training=False)
        
        # Il modello restituisce probabilità (Softmax è già presente)
        probs = preds[0].numpy()
        index = np.argmax(probs)
        confidenza = probs[index] * 100
        
        # Creazione dell'Embed
        embed = discord.Embed(title="🔍 Analisi CIFAR-10", color=0x3498db)
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Oggetto Identificato", value=f"✨ **{class_names[index].upper()}**", inline=False)
        embed.add_field(name="Grado di Sicurezza", value=f"📈 {confidenza:.2f}%", inline=True)
        
        # Diagnostica per il Check-sum dei pixel (deve variare tra immagini diverse)
        p_sum = np.sum(processed_img)
        embed.set_footer(text=f"Check-sum: {p_sum:.2f} | Mode: Inference")

        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await ctx.send(f"❌ Errore critico: {e}")

### Stato Personalizzato
bot mostra cosa sta facendo sotto il suo nome nella lista utenti.

In [ ]:
@bot.event
async def on_ready():
    # Imposta lo stato: "In ascolto di !classifica"
    activity = discord.Activity(type=discord.ActivityType.listening, name="!classifica")
    await bot.change_presence(status=discord.Status.online, activity=activity)
    print(f'✅ Bot pronto: {bot.user}')

### Gestione Errori

In [ ]:
# 1. Configurazione Stato Personalizzato
@bot.event
async def on_ready():
    # Imposta lo stato: "In ascolto di !info"
    activity = discord.Activity(type=discord.ActivityType.listening, name="!info")
    await bot.change_presence(status=discord.Status.online, activity=activity)
    print(f'✅ Bot online come: {bot.user}')

# 2. Gestione Errori Globale
@bot.event
async def on_command_error(ctx, error):
    if isinstance(error, commands.CommandNotFound):
        await ctx.send("❓ Comando non riconosciuto. Scrivi `!info` per vedere cosa posso fare.")
    elif isinstance(error, commands.MissingRequiredArgument):
        await ctx.send("⚠️ Mancano degli argomenti nel comando.")
    else:
        print(f"Errore non gestito: {error}")

In [ ]:
from tensorflow.keras.datasets import cifar10

@bot.command()
async def test(ctx):
    try:
        # Carichiamo i dati originali per un test puro
        (_, _), (x_test_raw, y_test_raw) = cifar10.load_data()
        
        # Prendiamo un'immagine a caso (es. la numero 10, che è un aereo)
        sample_idx = 10 
        img_test = x_test_raw[sample_idx]
        true_label = class_names[y_test_raw[sample_idx][0]]
        
        # Preprocessing identico a quello del bot
        img_input = img_test.astype('float32') / 255.0
        img_input = np.expand_dims(img_input, axis=0)
        
        # Predizione con training=False
        preds = model(img_input, training=False)
        probs = preds[0].numpy()
        index = np.argmax(probs)
        
        await ctx.send(f"🧪 **Test Interno**:\n"
                       f"Etichetta Reale: `{true_label.upper()}`\n"
                       f"Predizione Modello: `{class_names[index].upper()}`\n"
                       f"Confidenza: `{probs[index]*100:.2f}%`")
        
    except Exception as e:
        await ctx.send(f"❌ Errore nel test: {e}")

### Comando Classifica con Barra

In [ ]:
'''@bot.command()
async def classifica(ctx):
    # Controllo allegati
    if not ctx.message.attachments:
        await ctx.send("⚠️ Per favore, carica un'immagine insieme al comando `!classifica`.")
        return

    attachment = ctx.message.attachments[0]
    
    # Filtro estensioni
    if not any(attachment.filename.lower().endswith(ext) for ext in ['png', 'jpg', 'jpeg']):
        await ctx.send("❌ Formato file non supportato. Usa solo JPG o PNG.")
        return

    # Inizio caricamento grafico
    loading_msg = await ctx.send("⌛ [░░░░░░░░░░] 0% - Ricezione immagine...")
    
    try:
        # Download (40%)
        image_bytes = await attachment.read()
        await loading_msg.edit(content="⌛ [████░░░░░░] 40% - Elaborazione pixel...")
        
        # Preprocessing (70%)
        processed_img = prepare_image(image_bytes)
        await loading_msg.edit(content="⌛ [███████░░░] 70% - Interrogazione Rete Neurale...")
        
        # Predizione (100%)
        preds = model.predict(processed_img)
        await loading_msg.edit(content="⌛ [██████████] 100% - Analisi completata!")
        
        # Calcolo risultati
        index = np.argmax(preds)
        accuracy = np.max(tf.nn.softmax(preds[0])) * 100
        label = class_names[index]

        # Creazione dell'Embed finale
        embed = discord.Embed(
            title="🔍 Risultato del Riconoscimento",
            description=f"Analisi completata per l'utente {ctx.author.mention}",
            color=0x3498db # Colore blu elegante
        )
        embed.set_thumbnail(url=attachment.url)
        embed.add_field(name="Oggetto Predetto", value=f"**{label.upper()}**", inline=True)
        embed.add_field(name="Grado di Sicurezza", value=f"{accuracy:.2f}%", inline=True)
        embed.set_footer(text="AI Model: CIFAR-10 CNN | UniBari Project")

        # Rimuove la barra e invia il risultato
        await loading_msg.delete()
        await ctx.send(embed=embed)

    except Exception as e:
        await loading_msg.edit(content=f"❌ Si è verificato un errore durante l'analisi: {e}")'''

### Comando Informativo
Aggiungiamo un comando per spiegare agli utenti quali sono le 10 categorie che il modello può riconoscere (dataset CIFAR-10) e come utilizzare il bot.

In [ ]:
@bot.command()
async def info(ctx):
    # Creiamo un messaggio formattato in modo elegante (Embed)
    descrizione = (
        "Ciao! Sono il Bot del progetto di Classificazione Immagini.\n"
        "Utilizzo una rete neurale (CNN) addestrata sul dataset **CIFAR-10**.\n\n"
        "**Cosa posso fare?**\n"
        "Se mi invii una foto e scrivi `!classifica`, proverò a capire cosa rappresenta.\n\n"
        "**Le mie 10 categorie sono:**\n"
        f"✈️ {class_names[0]}, 🚗 {class_names[1]}, 🐦 {class_names[2]}, 🐱 {class_names[3]}, 🦌 {class_names[4]},\n"
        f"🐶 {class_names[5]}, 🐸 {class_names[6]}, 🐴 {class_names[7]}, 🚢 {class_names[8]}, 🚛 {class_names[9]}.\n\n"
        "**Istruzioni:**\n"
        "Carica una foto e scrivi `!classifica` nel campo del commento!"
    )
    
    await ctx.send(descrizione)

# Esecuzione
Inserisci il tuo Token e avvia la cella. Il bot rimarrà attivo finché la cella è in esecuzione.

In [ ]:
bot.run(TOKEN)